In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt

In [18]:
data_5min = pd.read_parquet("../raw_data/data_binance.parquet")
data = pd.read_parquet("../resampled_data/resampled_data_4h.parquet")

In [19]:
data.head()

,RV,log_RV,volume,trades
timestamp,,,,
2021-02-01 00:00:00,0.023693,-3.742555,14187.035713,299118.0
2021-02-01 04:00:00,0.019157,-3.955107,15350.212491,371011.0
2021-02-01 08:00:00,0.019405,-3.942214,19521.507872,443925.0
2021-02-01 12:00:00,0.020218,-3.901160,17478.706597,321562.0
2021-02-01 16:00:00,0.015424,-4.171835,9133.519374,219926.0


In [20]:
data["log_rv_plus_t"] = data["log_RV"].shift(-1)

In [21]:
def add_har_components(df, target_col='log_RV'):

    d = df.copy()
    d = d.sort_index()

    d["rv_daily"] = d["log_RV"].rolling(6).mean()
    d["rv_weekly"] = d["log_RV"].rolling(42).mean()
    d["rv_monthly"] = d["log_RV"].rolling(180).mean()

    return d

data = add_har_components(data)

In [22]:
data

,RV,log_RV,volume,trades,log_rv_plus_t,rv_daily,rv_weekly,rv_monthly
timestamp,,,,,,,,
2021-02-01 00:00:00,0.023693,-3.742555,14187.035713,299118.0,-3.955107,NaN,NaN,NaN
2021-02-01 04:00:00,0.019157,-3.955107,15350.212491,371011.0,-3.942214,NaN,NaN,NaN
2021-02-01 08:00:00,0.019405,-3.942214,19521.507872,443925.0,-3.901160,NaN,NaN,NaN
2021-02-01 12:00:00,0.020218,-3.901160,17478.706597,321562.0,-4.171835,NaN,NaN,NaN
2021-02-01 16:00:00,0.015424,-4.171835,9133.519374,219926.0,-4.577376,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
2026-09-03 08:00:00,0.005245,-5.250480,1507.399290,514832.0,-4.200452,-4.905595,-5.044502,-5.227233
2026-09-03 12:00:00,0.014989,-4.200452,7602.379530,1460907.0,-4.505130,-4.889030,-5.038099,-5.223828
2026-09-03 16:00:00,0.011052,-4.505130,3813.871140,814420.0,-4.562696,-4.789871,-5.032947,-5.220238


In [24]:
data_5min

,timestamp,open,high,low,close,volume,trades,quote_asset_volume,taker_buy_base,taker_buy_quote
0,2021-02-01 00:00:00,33092.97,33106.33,32777.14,32869.04,487.613325,12171.0,1.606026e+07,179.774943,5.918124e+06
1,2021-02-01 00:05:00,32866.41,32868.46,32464.24,32576.60,727.310947,14776.0,2.373056e+07,283.236278,9.238524e+06
2,2021-02-01 00:10:00,32580.67,32663.48,32545.02,32591.86,373.355815,8163.0,1.217302e+07,139.934790,4.562684e+06
3,2021-02-01 00:15:00,32591.87,32646.40,32300.00,32466.21,554.782922,11846.0,1.800890e+07,299.117644,9.710702e+06
4,2021-02-01 00:20:00,32466.20,32574.30,32330.64,32426.49,439.945032,10034.0,1.427537e+07,246.950888,8.012313e+06
...,...,...,...,...,...,...,...,...,...,...
587591,2026-09-03 23:40:00,81315.36,81338.00,81209.84,81234.00,38.803800,12504.0,3.153139e+06,19.775490,1.606683e+06
587592,2026-09-03 23:45:00,81234.01,81253.94,81181.95,81230.17,18.195230,7768.0,1.477817e+06,7.025440,5.706207e+05
587593,2026-09-03 23:50:00,81230.17,81328.76,81230.17,81295.64,39.661110,13208.0,3.224050e+06,19.455110,1.581405e+06
587594,2026-09-03 23:55:00,81295.63,81295.64,81239.79,81270.37,26.842480,5817.0,2.181360e+06,14.976460,1.217048e+06


In [28]:
def get_rq_4h_log(df):
    d = df.copy()
    d = d.set_index("timestamp")

    d["log_ret"] = np.log(d["close"]).diff()
    d = d.dropna(subset=["log_ret"])

    resampled = d["log_ret"].resample("4h")

    n_obs = resampled.count()

    sum_r4 = resampled.apply(lambda x: (x**4).sum())

    rq_values = (n_obs / 3) * sum_r4

    data = rq_values.to_frame(name="RQ")

    data["log_RQ"] = np.log(data["RQ"])
    data["sqrt_RQ"] = np.sqrt(data["RQ"])
    data["log_sqrt_RQ"] = np.log(data["sqrt_RQ"])

    data = data.replace([np.inf, -np.inf], np.nan).dropna()

    return data

rq = get_rq_4h_log(data_5min)

In [29]:
data = data.join(rq, how = "inner")

In [38]:
def add_jump_component(df_5min, df_4hour, price_col='close'):
    d_5m = df_5min.copy()
    d_4h = df_4hour.copy()

    d_5m = d_5m.set_index('timestamp')

    d_5m['return'] = np.log(d_5m[price_col] / d_5m[price_col].shift(1))

    d_5m['abs_ret'] = d_5m['return'].abs()
    d_5m['abs_ret_lag'] = d_5m['abs_ret'].shift(1)

    d_5m['bv_component'] = d_5m['abs_ret'] * d_5m['abs_ret_lag']

    pi_factor = np.pi / 2
    bv_4h = d_5m['bv_component'].resample('4h').sum() * pi_factor

    d_4h['BV'] = bv_4h

    d_4h['Jump'] = np.maximum(d_4h['RV'] - d_4h['BV'], 0)

    d_4h['log_Jump'] = np.log(d_4h['Jump'] + 1)

    d_4h = d_4h.drop(columns=['BV'])

    return d_4h

In [39]:
data = add_jump_component(data_5min, data, price_col='close')

In [41]:
def add_pump_risk(df_5min, df_4hour):
    d_5m = df_5min.copy()
    d_4h = df_4hour.copy()

    d_5m = d_5m.set_index('timestamp')

    if "log_ret" not in d_5m.columns:
        d_5m["log_ret"] = np.log(d_5m["close"] / d_5m["close"].shift(1))

    vol_4h = d_5m["volume"].resample("4h").sum()
    max_ret_4h = d_5m["log_ret"].resample("4h").max()

    d_4h["volume"] = vol_4h.reindex(d_4h.index)
    d_4h["max_ret"] = max_ret_4h.reindex(d_4h.index)

    vol_mean_180 = d_4h["volume"].shift(1).rolling(180, min_periods=180).mean()
    vol_std_180 = d_4h["volume"].shift(1).rolling(180, min_periods=180).std()

    d_4h["vol_z_30d"] = (d_4h["volume"] - vol_mean_180) / (vol_std_180 + 1e-8)

    ret_mean_180 = d_4h["max_ret"].shift(1).rolling(180, min_periods=180).mean()
    ret_std_180 = d_4h["max_ret"].shift(1).rolling(180, min_periods=180).std()
    d_4h["ret_z_30d"] = (d_4h["max_ret"] - ret_mean_180) / (ret_std_180 + 1e-8)

    d_4h["pump_risk"] = d_4h["vol_z_30d"] + d_4h["ret_z_30d"]

    d_4h = d_4h.drop(columns=["volume", "max_ret", "ret_z_30d"])

    return d_4h

In [42]:
data = add_pump_risk(data_5min, data)

In [43]:
data.to_parquet("cleaned_data.parquet")